In [1]:
from gradio_client import Client   # for api use

import ast                         # for text to list conversion (make data useful)
import pandas as pd                # for easy data use; you can use polars or whatever you like

# start 8:14pm

# Accessing the API

Accessing the API is simple. You pass your key to the client, and then you will see a message the API has loaded.

In [2]:
token = '../api_creds/vi_key_1.csv'

# load my key
df = pd.read_csv(token) # easy and lazy lol; using pandas anyway ;p
api_key = df['key'][0]  # easy and lazy lol; using pandas anyway ;p

# connect to api
client = Client("verdantintel/gsv3", token=api_key)

Loaded as API: https://verdantintel-gsv3.hf.space


# Helper Function

In [3]:
def parse_events(text):

    events = []

    for block in text.strip().split("\n\n"):
        lines = block.split("\n")

        event = {"title": lines[0].strip()}

        for line in lines[1:]:
            if ":" in line:
                key, value = line.split(":", 1)
                key = key.strip().lower().replace(" ", "_")
                value = value.strip()
                event[key] = value

        events.append(event)

    return events

In [4]:
def ask_api(client, question):

    answer = client.predict(question=question, api_name="/predict")
    
    #answer = ast.literal_eval(answer)
    events = parse_events(answer)

    return events

    #return answer

# Searching for Data

In [5]:
question = 'what is happening in hillsboro, beaverton, or portland on March 21-23, 2026?'

answer = ask_api(client, question)
len(answer)

42

In [6]:
answer[0]

{'title': 'Spring Fest on the Mountain',
 'performers': 'Garcia Birthday Band',
 'event_dates': 'March 21st & 22nd, 2026 at 1:00 PM',
 'price': 'Not provided in source.',
 'venue': 'Mt. Hood Skibowl',
 'location': 'Portland, OR',
 'contact_information': 'Not provided in source.',
 'description': 'Join us for the 4th Annual Spring Fest on the Mountain featuring free live music, great food, beer, and activities such as the Smooth Moves Rail Jam and Cosmic Tubing at Mt. Hood Skibowl.',
 'genres': 'Not provided in source.',
 'source_url': 'https://dopdx.com/events/2026/3/21/spring-fest-on-the-mountain-tickets'}

# Methodology

- Fetch GrooveSeeker events (done)
- Read quickly with my eyes looking for interesting things
- Reduce to a few options that I am actually interest in
- Go have fun

The test of GrooveSeeker is whether or not it leads to things that I enjoy. The technical tests are already done.

In [7]:
answer[0:2]

[{'title': 'WonderLab: Pages of Us // A DIY Family Coloring Zine Workshop',
  'performers': 'Erika Rier',
  'event_dates': 'March 21, 2026 10:00 am - 2:30 pm',
  'price': '$40 for 1 adult & 1 child, additional adult $10, additional child $5 each',
  'venue': 'PAM CUT',
  'location': '934 SW Salmon St., Portland, OR',
  'contact_information': 'info@pamcut.org, 503-221-1156',
  'description': 'Join teaching artist Erika Rier for a hands-on family workshop where families create their own zine based on everyday adventures, combining art-making and storytelling.',
  'genres': 'Not provided in source.',
  'source_url': 'https://portlandartmuseum.org/event/wonderlab-pages-of-us-a-diy-family-coloring-zine-workshop/'},
 {'title': 'Mundo de las Mujeres 2026',
  'performers': 'Erlina Ortiz, Alexis Scheer, Mia Torres, Andrea Menchaca Hall',
  'event_dates': 'Multiple dates through March 21, various times',
  'price': 'Pay-What-You-Will Pricing – Starting at $5',
  'venue': 'Milagro Theatre',
  'lo

Let's loop through titles and see what looks interesting. I don't want to be distracted by the other context yet. Title and descriptiona re good enough.

In [8]:
df = pd.DataFrame(answer)

df = df[df['source_url'].str.startswith('https')] # simple URL filter; drop empties; the rest are https

df.drop_duplicates(inplace=True)
df.sort_values('title', inplace=True)

for row in df.iterrows():
    
    title = row[1]['title']
    description = row[1]['description']
    event_dates = row[1]['event_dates']
    
    url = row[1]['source_url']
    
    print(title)
    print()
    print(event_dates)
    print()
    print(description)
    print()
    print(url)
    print()
    print('-------------------------------------------------')
    print()

A brand new sketch comedy show from The Sisters of Mercy

March 21, 2026, 8:00 PM

Join The Sisters of Mercy for a new sketch comedy show, "JAM - PACKED," featuring brand new sketches. The quartet has been entertaining audiences since 2019 and often sells out their shows.

https://www.merctickets.com/events/182973325/a-brand-new-sketch-comedy-show-from-the-sisters-of-mercy

-------------------------------------------------

Almost Famous Crafternoon w/ Ritual Dyes

March 22, 2026 4:00 pm - 6:00 pm

A crafting event hosted by the Portland Art Museum featuring Ritual Dyes.

https://portlandartmuseum.org/event/almost-famous-crafternoon-w-ritual-dyes/

-------------------------------------------------

BROTHER RUCKUS

Sun March 22, 2026 (Doors: 6:30 pm)

Brother Ruckus, a drummer and alumnus of Berklee College of Music, will perform at Jack London Revue. He is known for high-energy performances and has shared the stage with many acclaimed artists, aiming to inspire young musicians.

https:

Since writing the article, I often come back to this notebook to look up fun events. 

Read the article: https://100daysofnetworks.substack.com/p/day-81-of-100daysofnetworks